In [0]:
# Leer tabla Silver
df_silver = spark.table("bigdata.default.silver_sales_cleaned")
print("Registros en Silver:", df_silver.count())
df_silver.show(5)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# Calcular fecha de referencia (dia siguiente a la ultima transaccion)
snapshot_date_str = (
    df_silver
    .agg(F.max("fecha_factura").alias("max_date"))
    .withColumn("snapshot_date", F.date_add(F.col("max_date"), 1))
    .collect()[0]["snapshot_date"]
)
print("Snapshot date:", snapshot_date_str)

In [0]:
snapshot_lit = F.lit(snapshot_date_str)

# Calcular metricas RFM por cliente
# Recency   -> dias desde la ultima compra
# Frequency -> numero de facturas unicas
# Monetary  -> gasto total acumulado
df_rfm = (
    df_silver
    .groupBy("id_cliente")
    .agg(
        F.datediff(snapshot_lit, F.max("fecha_factura")).cast(IntegerType()).alias("recency"),
        F.countDistinct("numero_factura").cast(IntegerType()).alias("frequency"),
        F.round(F.sum("TotalAmount"), 2).cast(DoubleType()).alias("monetary"),
        F.max("fecha_factura").alias("ultima_compra"),
        F.first("pais").alias("pais")
    )
)
print("Clientes unicos procesados:", df_rfm.count())
df_rfm.show(5)

In [0]:
# Estadisticas descriptivas de las metricas RFM
df_rfm.select("recency", "frequency", "monetary").describe().show()

In [0]:
# Agregar metadatos y guardar tabla Gold
df_gold = df_rfm.withColumn("gold_timestamp", F.current_timestamp())

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bigdata.default.gold_customer_features")

print("gold_customer_features guardada!")

In [0]:
bronze = spark.table("bigdata.default.bronze_sales_raw").count()
silver = spark.table("bigdata.default.silver_sales_cleaned").count()
gold   = spark.table("bigdata.default.gold_customer_features").count()

print(f"Bronze: {bronze:,} registros")
print(f"Silver: {silver:,} registros")
print(f"Gold:   {gold:,} clientes con RFM")